# Week 4: Hamiltonian Simulation Workflow

This notebook covers the Hamiltonian Simulation Workflow, consistent with the Qiskit Pattern and IBM Quantum's function templates.
We use local simulation via `AerSimulator` for hands-on practice without requiring an IBM Quantum account.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector, Operator
from qiskit.synthesis import SuzukiTrotter, LieTrotter
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit.primitives import BackendEstimatorV2
import numpy as np
import matplotlib.pyplot as plt

backend = AerSimulator()
print(f'Using backend: {backend.name}')

## Theory — Hamiltonian Simulation

- **What is a Hamiltonian?** In quantum mechanics, a Hamiltonian is an observable that corresponds to the total energy of the system and serves as the generator of time evolution.
- **Time evolution operator:** The state of a quantum system evolves over time according to the operator $U(t) = e^{-iHt}$.
- **Why quantum computers?** Simulating quantum systems is a natural fit for quantum hardware, avoiding the exponential classical memory overhead required to store statevectors.
- **The Trotterization approach:** Since $H$ is typically a sum of non-commuting terms $H = H_1 + H_2 + \dots$, we can't just exponentiate each term independently. We approximate the evolution using Trotterization, splitting the time evolution into small time steps $\Delta t$ and approximating $e^{-iHt} \approx (e^{-iH_1 \Delta t} e^{-iH_2 \Delta t} \dots)^n$.
- **First-order vs Second-order:** Lie-Trotter is a first-order formula. Suzuki-Trotter is a symmetric second-order formula that achieves better accuracy for the same step size.
- **Trotter error:** $O(t^2/n)$ for first-order (Lie-Trotter), $O(t^3/n^2)$ for second-order (Suzuki-Trotter).
- **Depth vs accuracy tradeoff:** More Trotter steps ($n$) means lower theoretical error, but a deeper circuit which increases noise when run on real hardware.

## Step 1 — Map (Define Hamiltonian & Build Trotter Circuit)

We define a 4-qubit Heisenberg XXZ spin chain Hamiltonian:

$H = \sum_{i} [J_x(X_i X_{i+1}) + J_y(Y_i Y_{i+1}) + J_z(Z_i Z_{i+1})] + \sum_i h_i Z_i$

with $J_x = J_y = 1.0$, $J_z = 0.5$ (anisotropy), and small random fields $h_i$.

In [ ]:
# Define a 4-qubit Heisenberg XXZ Hamiltonian
num_qubits = 4
J_x, J_y, J_z = 1.0, 1.0, 0.5  # coupling constants (XXZ anisotropy)

# Build the Hamiltonian as a SparsePauliOp
pauli_list = []
coeffs = []

for i in range(num_qubits - 1):
    # XX interaction
    xx = ['I'] * num_qubits
    xx[i], xx[i+1] = 'X', 'X'
    pauli_list.append(''.join(xx))
    coeffs.append(J_x)
    
    # YY interaction
    yy = ['I'] * num_qubits
    yy[i], yy[i+1] = 'Y', 'Y'
    pauli_list.append(''.join(yy))
    coeffs.append(J_y)
    
    # ZZ interaction
    zz = ['I'] * num_qubits
    zz[i], zz[i+1] = 'Z', 'Z'
    pauli_list.append(''.join(zz))
    coeffs.append(J_z)

# Add small random longitudinal fields
np.random.seed(42)
for i in range(num_qubits):
    z_field = ['I'] * num_qubits
    z_field[i] = 'Z'
    pauli_list.append(''.join(z_field))
    coeffs.append(0.1 * np.random.randn())

hamiltonian = SparsePauliOp(pauli_list, coeffs=coeffs)
print(f"Hamiltonian has {len(hamiltonian)} Pauli terms on {num_qubits} qubits")
print("\nHamiltonian terms:")
for label, coeff in zip(hamiltonian.paulis.to_labels(), hamiltonian.coeffs):
    print(f"  {coeff:+.4f} * {label}")

In [ ]:
# Define observables to measure
# We'll track the magnetization on each qubit: <Z_i>
observables = []
for i in range(num_qubits):
    z_obs = ['I'] * num_qubits
    z_obs[i] = 'Z'
    observables.append(SparsePauliOp(''.join(z_obs)))

print(f"Tracking {len(observables)} observables (single-qubit Z magnetization)")

In [ ]:
# Build time evolution circuits using Suzuki-Trotter decomposition
evolution_time = 1.0
num_trotter_steps = 4

# Create the PauliEvolutionGate
evo_gate = PauliEvolutionGate(
    hamiltonian,
    time=evolution_time,
    synthesis=SuzukiTrotter(reps=num_trotter_steps)
)

# Build the full circuit with an initial state
# Start with |↑↓↑↓⟩ = |0101⟩ (Néel state, common for spin chains)
initial_state = QuantumCircuit(num_qubits)
for i in range(1, num_qubits, 2):
    initial_state.x(i)  # flip every other qubit

trotter_circuit = initial_state.copy()
trotter_circuit.append(evo_gate, range(num_qubits))

print(f"Trotter circuit: {trotter_circuit.num_qubits} qubits")
print(f"Evolution time: {evolution_time}")
print(f"Trotter steps: {num_trotter_steps}")

In [ ]:
# Decompose and visualize the Trotter circuit structure
decomposed = trotter_circuit.decompose().decompose()
print(f"Decomposed circuit depth: {decomposed.depth()}")
print(f"Gate counts: {dict(decomposed.count_ops())}")
decomposed.draw('mpl', fold=80)

In [ ]:
# Compare circuit depth vs number of Trotter steps
step_counts = [1, 2, 4, 8, 16]
depths = []
gate_counts_2q = []

for steps in step_counts:
    evo = PauliEvolutionGate(
        hamiltonian,
        time=evolution_time,
        synthesis=SuzukiTrotter(reps=steps)
    )
    qc = QuantumCircuit(num_qubits)
    qc.append(evo, range(num_qubits))
    dec = qc.decompose().decompose()
    depths.append(dec.depth())
    cx_count = dec.count_ops().get('cx', 0)
    gate_counts_2q.append(cx_count)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(step_counts, depths, 'bo-')
ax1.set_xlabel('Number of Trotter Steps')
ax1.set_ylabel('Circuit Depth')
ax1.set_title('Circuit Depth vs Trotter Steps')
ax1.grid(True, alpha=0.3)

ax2.plot(step_counts, gate_counts_2q, 'rs-')
ax2.set_xlabel('Number of Trotter Steps')
ax2.set_ylabel('Number of CX Gates')
ax2.set_title('Two-Qubit Gate Count vs Trotter Steps')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## AQC-Tensor Conceptual Overview

- **The problem:** Deep Trotter circuits have too many gates for noisy hardware, leading to accumulated errors.
- **AQC-Tensor solution:** Compress the initial Trotter layers using tensor network (MPS) methods.
- **Workflow:**
  1. Build a target circuit (high Trotter steps) and convert to Matrix Product State (MPS).
  2. Build an ansatz from a fewer-step Trotter circuit.
  3. Iteratively optimize ansatz parameters (using L-BFGS-B) to maximize fidelity with target MPS.
  4. Resulting circuit has fewer gates but approximates the same time evolution.
  5. Append remainder Trotter steps for the remaining evolution time.
- **Benefits:** Significant reduction in two-qubit gate depth $\to$ less noise on real hardware.
- **Key Qiskit addons used:** `qiskit-addon-aqc-tensor`, `qiskit-addon-utils`, `quimb` (tensor network library).

Workflow Diagram:
```
Target Circuit (many Trotter steps)
     │
     ▼ tensornetwork_from_circuit()
Target MPS
     │
     ▼ OneMinusFidelity objective
Optimized Ansatz (fewer gates)
     │
     ▼ + Remainder Trotter steps
Final Circuit (reduced depth)
```

## Step 2 — Optimize (Transpile for Hardware)

In [ ]:
# Step 2: Transpile the circuit for the backend
# Compare optimization levels
for opt_level in [0, 1, 2, 3]:
    pm = generate_preset_pass_manager(backend=backend, optimization_level=opt_level)
    isa_circuit = pm.run(trotter_circuit)
    depth = isa_circuit.depth()
    cx_count = isa_circuit.count_ops().get('cx', 0)
    print(f"Optimization level {opt_level}: depth={depth}, CX gates={cx_count}")

# Use level 3 for our experiment
pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
isa_circuit = pm.run(trotter_circuit)
print(f"\nFinal ISA circuit: depth={isa_circuit.depth()}, "
      f"CX gates={isa_circuit.count_ops().get('cx', 0)}")

In [ ]:
# Remap observables to match the transpiled circuit layout
isa_observables = [
    obs.apply_layout(isa_circuit.layout) for obs in observables
]
print("Observables remapped to ISA layout")
print(f"Original qubit count: {observables[0].num_qubits}")
print(f"ISA qubit count: {isa_observables[0].num_qubits}")

## Step 3 — Execute (Estimator)

We use the `BackendEstimatorV2` to evaluate our observables. On real hardware, you would typically enable error mitigation options such as Zero-Noise Extrapolation (ZNE), Pauli Twirling, or Measurement Error Mitigation via the estimator configuration.

In [ ]:
# Step 3: Execute using Estimator
# Run simulation for multiple evolution times
time_points = np.linspace(0, 2.0, 11)  # 0 to 2.0 in 11 steps
num_steps = 4  # Trotter steps per time point

estimator = BackendEstimatorV2(backend=backend)

all_magnetizations = {i: [] for i in range(num_qubits)}

print("Running Hamiltonian simulation...")
for t in time_points:
    if t == 0:
        # At t=0, measure the initial state directly
        init_qc = initial_state.copy()
        pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
        isa_init = pm.run(init_qc)
        isa_obs_init = [obs.apply_layout(isa_init.layout) for obs in observables]
        
        for i, obs in enumerate(isa_obs_init):
            job = estimator.run([(isa_init, obs)])
            result = job.result()
            all_magnetizations[i].append(result[0].data.evs)
    else:
        # Build and run Trotter circuit for this time
        evo_gate_t = PauliEvolutionGate(
            hamiltonian,
            time=t,
            synthesis=SuzukiTrotter(reps=num_steps)
        )
        qc_t = initial_state.copy()
        qc_t.append(evo_gate_t, range(num_qubits))
        
        pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
        isa_t = pm.run(qc_t)
        isa_obs_t = [obs.apply_layout(isa_t.layout) for obs in observables]
        
        for i, obs in enumerate(isa_obs_t):
            job = estimator.run([(isa_t, obs)])
            result = job.result()
            all_magnetizations[i].append(result[0].data.evs)
    
    print(f"  t={t:.2f} done")

print("Simulation complete!")

## Step 4 — Post-process & Visualize

In [ ]:
# Compute exact time evolution for comparison
# Using matrix exponentiation (feasible for 4 qubits = 16x16 matrix)
from scipy.linalg import expm

H_matrix = hamiltonian.to_matrix()

# Initial state |0101⟩ = Néel state
init_sv = Statevector.from_label('0101')

exact_magnetizations = {i: [] for i in range(num_qubits)}

for t in time_points:
    # Time-evolved state
    U = expm(-1j * H_matrix * t)
    evolved_state = Statevector(U @ init_sv.data)
    
    for i, obs in enumerate(observables):
        exp_val = evolved_state.expectation_value(obs).real
        exact_magnetizations[i].append(exp_val)

In [ ]:
# Plot: Trotter simulation vs exact results
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Hamiltonian Simulation: Qubit Magnetization ⟨Zᵢ⟩ vs Time', fontsize=14)

colors = ['#0f62fe', '#da1e28', '#198038', '#8a3ffc']

for i, ax in enumerate(axes.flat):
    ax.plot(time_points, exact_magnetizations[i], '-', 
            color=colors[i], linewidth=2, label='Exact')
    ax.plot(time_points, all_magnetizations[i], 'o', 
            color=colors[i], markersize=6, label=f'Trotter (n={num_steps})')
    ax.set_xlabel('Time t')
    ax.set_ylabel(f'⟨Z_{i}⟩')
    ax.set_title(f'Qubit {i}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-1.1, 1.1)

plt.tight_layout()
plt.show()

In [ ]:
# Trotter error analysis: compare different step counts
fig, ax = plt.subplots(figsize=(8, 5))

t_eval = 1.0  # Evaluate error at t=1.0
step_list = [1, 2, 4, 8, 16, 32]
errors = []

# Exact at t=1.0
U_exact = expm(-1j * H_matrix * t_eval)
exact_sv = Statevector(U_exact @ init_sv.data)
exact_z0 = exact_sv.expectation_value(observables[0]).real

for steps in step_list:
    evo = PauliEvolutionGate(
        hamiltonian,
        time=t_eval,
        synthesis=SuzukiTrotter(reps=steps)
    )
    qc = initial_state.copy()
    qc.append(evo, range(num_qubits))
    
    # Use statevector simulation for exact Trotter result
    sv = Statevector.from_instruction(qc)
    trotter_z0 = sv.expectation_value(observables[0]).real
    errors.append(abs(trotter_z0 - exact_z0))

ax.loglog(step_list, errors, 'bo-', linewidth=2, markersize=8, label='Measured error')

# Theoretical scaling: O(1/n²) for Suzuki-Trotter
if errors[0] > 0:
    ref = errors[0] * (step_list[0] / np.array(step_list))**2
    ax.loglog(step_list, ref, 'r--', alpha=0.5, label='O(1/n²) reference')

ax.set_xlabel('Number of Trotter Steps (n)')
ax.set_ylabel('|⟨Z₀⟩_Trotter - ⟨Z₀⟩_exact|')
ax.set_title(f'Suzuki-Trotter Error Convergence (t={t_eval})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Qiskit Functions & Serverless (Conceptual)

- **Qiskit Function template pattern:** Write a reusable program that encapsulates the entire workflow.
- **Key APIs:** `get_arguments()`, `save_result()` from `qiskit_serverless`.
- `%%writefile` magic can be used to save template code to a Python file.
- **Deployment:** Deploy to IBM Quantum Platform via `QiskitServerless` and `QiskitFunction`.
- **Running remotely:** `template.run(arguments)` with domain-specific inputs.
- **The dry_run option:** Useful for inspecting circuit depth before hardware execution.
- **Error mitigation:** Options used in the template include ZNE (gate folding), measurement mitigation, and twirling.
- **Benefits:** Encapsulate complex workflows, reuse across different Hamiltonians, and run on real QPUs.

### Key Takeaways

1. Hamiltonian simulation maps naturally to quantum circuits via Trotterization.
2. Suzuki-Trotter provides second-order error convergence $O(t^3/n^2)$.
3. AQC-Tensor compresses deep Trotter circuits using tensor network optimization.
4. The Qiskit Pattern (Map→Optimize→Execute→Post-process) structures the entire workflow.
5. Qiskit Functions enable packaging and remote deployment of simulation workflows.
6. Error mitigation (ZNE, twirling, measurement mitigation) is essential for real hardware.